# LightGBM LambdaRank · Pau

Objetivo: mejorar el LightGBM de referencia (CV ≈ 0.653 con las columnas originales).

**Reglas (EQUIPO.md):**
- Cada snapshot se predice **solo con sus propias filas**: toda feature nueva se calcula con `groupby("sample_id")` (o un grupo más fino dentro del snapshot).
- Validación: los 5 folds de `Data/folds.csv` (por controlador). Se decide por la CV, no por Kaggle.
- Una feature solo entra si **mejora la CV**.

**Cómo usar el notebook:** ejecuta de arriba abajo una vez. Después, para probar algo, cambia la sección 3 (features) o `PARAMS` y vuelve a ejecutar la celda de experimentos. Cada experimento se apunta en `experiments/lgb_runs.csv`.

## 1. Configuración

In [16]:
import subprocess, sys, time
import numpy as np, pandas as pd, lightgbm as lgb
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
for p in ("oof", "preds", "submissions", "models", "experiments"):
    (ROOT / p).mkdir(exist_ok=True)

NAME = "lgb_coord"   # nombre de los archivos de salida
TYPES = ['entry_conflict_resolution', 'exit_conflict_resolution', 'entry_coordination', 'exit_coordination',
         'assume_aircraft', 'transfer_aircraft', 'conflict_resolution', 'non_conformance_resolution',
         'qos_improvement', 'entry_conditions', 'exit_conditions', 'aircraft_request', 'zone_conflict',
         'return_to_route']

# Parámetros base de EQUIPO.md
PARAMS = dict(objective="lambdarank", n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=50,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, lambdarank_truncation_level=5,
              random_state=0, verbose=-1)

## 2. Datos
- `lab` = train + validation (con etiqueta), cada fila con su `fold` (0–4).
- `test` = test, sin etiqueta.
- `cs` = callsign del avión de la tarea (sacado de `task_name`). **No es una feature**: solo sirve para agrupar las tareas de un mismo avión.

In [17]:
def load(name):
    d = pd.read_csv(DATA / f"{name}.csv")
    pref = d.task_index.map(lambda t: len(TYPES[t]) + 1)
    d["cs"] = [n[p:].split("_")[0] for n, p in zip(d.task_name, pref)]
    return d

lab = pd.concat([load("train"), load("validation")], ignore_index=True)
lab = lab.merge(pd.read_csv(DATA / "folds.csv"), on="atco", how="left")
test = load("test")
lab = lab.sort_values(["sample_id", "row_index"]).reset_index(drop=True)
test = test.sort_values(["sample_id", "row_index"]).reset_index(drop=True)

assert lab.fold.notna().all(), "hay controladores sin fold"
assert (lab.groupby("sample_id").target_index.sum() == 1).all(), "alguna muestra no tiene exactamente 1 respuesta"
print(f"lab: {lab.sample_id.nunique():,} muestras, {len(lab):,} filas | test: {test.sample_id.nunique():,} muestras")
print(lab.drop_duplicates("sample_id").groupby("fold").atco.unique())

lab: 13,175 muestras, 474,688 filas | test: 1,863 muestras
fold
0        [7, 14, 18]
1      [1, 3, 4, 15]
2      [2, 6, 9, 16]
3        [8, 17, 19]
4    [5, 10, 11, 20]
Name: atco, dtype: object


## 3. Features
Cada grupo es una función que recibe el dataframe y devuelve **solo las columnas nuevas**. Todas agrupan por `sample_id`, así que solo usan el propio snapshot.

| Grupo | Qué añade |
|---|---|
| `snapshot` | Ranking y diferencia respecto al mínimo **dentro del snapshot** de varias columnas de urgencia/actividad |
| `tipo` | Los mismos rankings **dentro de las tareas del mismo tipo** del snapshot |
| `avion` | Si la tarea es la más antigua de su avión, cuántos tipos distintos tiene el avión |
| `tipo_estado` | Cruces tipo × estado: assume con oferta pendiente, transfer debido, zona con su distancia, etc. |
| `contexto` | Cómo está el snapshot: nº de aviones, nº de transfers/assumes/zonas/conflictos pendientes |

Para añadir una idea nueva: escribe otra función `fe_...` y añádela a `FEATURE_GROUPS`.

In [18]:
META = ['sample_id', 'target_index', 'session', 'atco', 'scenario', 'timestamp', 'task_name', 'row_index',
        'fold', 'cs']
BASE = [c for c in lab.columns if c not in META]
RANK_COLS = ["a_since_flight", "x_task_age", "x_state_age", "time_to_MD", "k_timeToExitPoint",
             "distance_to_zone", "abs_tts"]


def _with_abs(df):
    return df.assign(abs_tts=df.time_to_sector.abs())


def fe_snapshot(df):
    x = _with_abs(df)
    g = x.groupby("sample_id")
    out = {}
    for c in RANK_COLS:
        out[f"rs_{c}"] = g[c].rank(method="min")
        out[f"ds_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_tipo(df):
    x = _with_abs(df)
    g = x.groupby(["sample_id", "task_index"])
    out = {}
    for c in RANK_COLS:
        out[f"rt_{c}"] = g[c].rank(method="min")
        out[f"dt_{c}"] = x[c] - g[c].transform("min")
    return pd.DataFrame(out, index=df.index)


def fe_avion(df):
    g = df.groupby(["sample_id", "cs"])
    out = {
        "ac_age_rank": g.x_task_age.rank(method="min", ascending=False),   # 1 = la tarea más antigua del avión
        "ac_n_types": g.task_index.transform("nunique"),
        "ac_has_assume": g.task_index.transform(lambda s: (s == 4).any()).astype(int),
    }
    return pd.DataFrame(out, index=df.index)


def fe_tipo_estado(df):
    t = df.task_index
    out = {
        "ts_assume_offered": ((t == 4) & (df.x_transfer_in == 1)).astype(int),
        "ts_transfer_due": ((t == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "ts_entry_coord_done": ((t == 2) & (df.x_isc_coordinated == 1)).astype(int),
        "ts_entry_notified": ((t == 2) & (df.x_isc_notified == 1)).astype(int),
        "ts_zone_dist": df.distance_to_zone.where(t == 12),
        "ts_conflict_ttmd": df.time_to_MD.where(t.isin([0, 1, 6])),
        "ts_twin_assume": ((t == 4) & (df.l_twin_AE == 1)).astype(int),
        "ts_twin_entry": ((t == 2) & (df.l_twin_AE == 1)).astype(int),
    }
    return pd.DataFrame(out, index=df.index)


def fe_contexto(df):
    g = df.groupby("sample_id")
    tmp = pd.DataFrame({
        "sample_id": df.sample_id,
        "is_assume_offer": ((df.task_index == 4) & (df.x_transfer_in == 1)).astype(int),
        "is_transfer_due": ((df.task_index == 5) & (df.k_aircraftNeedsToBeTransferred == 1)).astype(int),
        "is_zone": (df.task_index == 12).astype(int),
        "is_conflict": df.task_index.isin([0, 1, 6]).astype(int),
        "is_entry_coord": (df.task_index == 2).astype(int),
    })
    gt = tmp.groupby("sample_id")
    out = {"cx_n_aircraft": g.cs.transform("nunique")}
    for c in tmp.columns[1:]:
        out["cx_n_" + c[3:]] = gt[c].transform("sum")
    return pd.DataFrame(out, index=df.index)


def fe_coord_salida(df):
    """
    Features específicas de exit_coordination (task_index == 3).
    Todas se construyen a partir del snapshot actual.
    """
    es_ec = df["task_index"].eq(3)
    out = {}

    # Estados que pueden indicar si hay que coordinar
    # o si la tarea está más relacionada con la transferencia.
    estados = {
        "mismatch": "k_outCoordMismatch",
        "coord_tool": "k_coordinationToolActivation",
        "notified": "k_out_NOTIFIED",
        "coordinated": "k_out_COORDINATED",
        "transfer_tool": "k_transferToolActivation",
        "transfer_due": "k_aircraftNeedsToBeTransferred",
        "exit_met": "k_exitConditionsAreMet",
    }

    for nombre, columna in estados.items():
        valor = df[columna]

        # Interacción: el estado está activo en una tarea
        # de coordinación de salida.
        out[f"ec_{nombre}_activo"] = (
            es_ec & valor.eq(1)
        ).astype("int8")

        # No confundir un estado desconocido con un cero.
        out[f"ec_{nombre}_ausente"] = (
            es_ec & valor.isna()
        ).astype("int8")

    # Número de mensajes de coordinación intercambiados.
    revision = pd.to_numeric(
        df["k_outCoordRevision"], errors="coerce"
    )
    out["ec_revision"] = revision.where(es_ec)

    # Versión logarítmica para reducir el peso de valores altos.
    out["ec_revision_log"] = (
        np.log1p(revision.clip(lower=0)).where(es_ec)
    )

    # Tiempo hasta el punto de salida, específico de esta tarea.
    out["ec_tiempo_salida"] = (
        df["k_timeToExitPoint"].where(es_ec)
    )

    return pd.DataFrame(out, index=df.index)


def fe_return_route(df):
    """
    Features específicas de return_to_route (task_index == 13).
    """
    es_rtr = df["task_index"].eq(13)

    desviacion = pd.to_numeric(
        df["non_conformance_agreed_route_deviation_horizontally"],
        errors="coerce"
    )

    # Dejar la desviación solo en las tareas de volver a la ruta.
    dev_rtr = desviacion.where(es_rtr)

    out = {
        # Magnitud original en metros.
        "rtr_desviacion_m": dev_rtr,

        # Transformación logarítmica para reducir el efecto
        # de desviaciones extremadamente grandes.
        "rtr_desviacion_log": (
            np.log1p(desviacion.clip(lower=0)).where(es_rtr)
        ),

        # Distinguir si existe una desviación distinta de cero.
        "rtr_desviacion_positiva": (
            es_rtr & desviacion.gt(0)
        ).astype("int8"),

        # Detectar valores ausentes dentro de la tarea.
        "rtr_desviacion_ausente": (
            es_rtr & desviacion.isna()
        ).astype("int8"),

        # El diccionario advierte que algunos valores >100 km
        # pueden ser errores de datos.
        "rtr_desviacion_extrema": (
            es_rtr & desviacion.gt(100_000)
        ).astype("int8"),

        # Indicador de que se ha activado una regla de conformidad.
        "rtr_monitorizacion_activa": (
            es_rtr & df["monitoring_rule_activated"].eq(1)
        ).astype("int8"),

        "rtr_monitorizacion_ausente": (
            es_rtr & df["monitoring_rule_activated"].isna()
        ).astype("int8"),
    }

    # Posición de la desviación entre las tareas return_to_route
    # del mismo snapshot. 1 = desviación más grande.
    out["rtr_rango_desviacion"] = (
        dev_rtr.groupby(df["sample_id"])
        .rank(method="min", ascending=False)
    )

    return pd.DataFrame(out, index=df.index)


FEATURE_GROUPS = {
    "snapshot": fe_snapshot,
    "tipo": fe_tipo,
    "avion": fe_avion,
    "tipo_estado": fe_tipo_estado,
    "contexto": fe_contexto,
    "coord_salida": fe_coord_salida,
    "return_route": fe_return_route,
}



def build(df, groups):
    """Devuelve (df con las columnas nuevas, lista de features a usar)."""
    new = [FEATURE_GROUPS[g](df) for g in groups]
    out = pd.concat([df] + new, axis=1)
    return out, BASE + [c for n in new for c in n.columns]

## 4. Entrenamiento y CV
- `fit`: entrena un `LGBMRanker`. Le dice a LightGBM qué filas forman cada snapshot (`group`) para que aprenda a **ordenar dentro del snapshot**.
- `hit3`: para cada snapshot, 1 si la respuesta está entre las 3 de mayor score.
- `cv`: para cada fold, entrena con los otros 4 y predice ese. El resultado es el **OOF** (out-of-fold): una predicción para cada fila hecha por un modelo que no la vio.

In [19]:
def fit(A, F, params):
    m = lgb.LGBMRanker(**params)
    m.fit(A[F], A.target_index, group=A.groupby("sample_id", sort=False).size().values,
          categorical_feature=["task_index"])
    return m


def hit3(d, score):
    t = d[["sample_id", "target_index"]].assign(s=np.asarray(score))
    t = t.sort_values(["sample_id", "s"], ascending=[True, False])
    return t.groupby("sample_id").head(3).groupby("sample_id").target_index.max()


def cv(d, F, params):
    oof = np.full(len(d), np.nan)
    for f in range(5):
        tr, va = (d.fold != f).values, (d.fold == f).values
        oof[va] = fit(d[tr], F, params).predict(d.loc[va, F])
    h = hit3(d, oof)
    fold_of = d.drop_duplicates("sample_id").set_index("sample_id").fold
    return oof, h, h.groupby(fold_of.reindex(h.index)).mean()


LOG = ROOT / "experiments" / "lgb_runs.csv"


def experimento(nombre, groups, params=None, nota=""):
    params = {**PARAMS, **(params or {})}
    t0 = time.time()
    d, F = build(lab, groups)
    oof, h, por_fold = cv(d, F, params)
    fila = dict(hora=time.strftime("%H:%M"), nombre=nombre, grupos="+".join(groups) or "-",
                params_cambiados=str({k: v for k, v in params.items() if PARAMS.get(k) != v}),
                n_features=len(F), cv_top3=round(h.mean(), 4), std_folds=round(por_fold.std(), 4), nota=nota)
    pd.DataFrame([fila]).to_csv(LOG, mode="a", header=not LOG.exists(), index=False)
    print(f"{nombre:28s} CV top-3 = {h.mean():.4f}  (folds: {' '.join(f'{v:.3f}' for v in por_fold)})"
          f"  {len(F)} features  {time.time() - t0:.0f}s")
    return oof, h

## 5. Experimentos
Primero la referencia (columnas originales). Después cada grupo de features **por separado**, para ver cuál aporta. Por último, todos juntos.

Diferencias menores de ~0.003 pueden ser ruido: fíjate también en si mejora en la mayoría de folds.

In [20]:

# Referencia
o_ctx, h_ctx = experimento(
    "ref_contexto", ["contexto"]
)

# Características que ya existían, combinadas con contexto
o_ctx_tipo, h_ctx_tipo = experimento(
    "contexto+tipo", ["contexto", "tipo"]
)

o_ctx_estado, h_ctx_estado = experimento(
    "contexto+tipo_estado", ["contexto", "tipo_estado"]
)

# Nuevas características que estamos investigando
o_ctx_coord, h_ctx_coord = experimento(
    "contexto+coord_salida",
    ["contexto", "coord_salida"]
)

o_ctx_ruta, h_ctx_ruta = experimento(
    "contexto+return_route",
    ["contexto", "return_route"]
)

# Combinación de las dos nuevas ideas
o_ctx_ambos, h_ctx_ambos = experimento(
    "contexto+coord+ruta",
    ["contexto", "coord_salida", "return_route"]
)


ref_contexto                 CV top-3 = 0.6550  (folds: 0.689 0.657 0.659 0.627 0.644)  94 features  82s
contexto+tipo                CV top-3 = 0.6504  (folds: 0.680 0.661 0.645 0.623 0.643)  108 features  95s
contexto+tipo_estado         CV top-3 = 0.6486  (folds: 0.673 0.669 0.640 0.620 0.640)  102 features  82s
contexto+coord_salida        CV top-3 = 0.6522  (folds: 0.687 0.655 0.643 0.640 0.638)  111 features  79s
contexto+return_route        CV top-3 = 0.6493  (folds: 0.677 0.657 0.646 0.629 0.637)  102 features  85s
contexto+coord+ruta          CV top-3 = 0.6556  (folds: 0.690 0.659 0.645 0.630 0.654)  119 features  94s


In [21]:
# Historial de todos los experimentos (el más reciente al final)
pd.read_csv(LOG).tail(15)

,hora,nombre,grupos,params_cambiados,n_features,cv_top3,std_folds,nota
28,21:46,contexto+coord_salida,contexto+coord_salida,{},111,0.6522,0.0201,NaN
29,21:48,contexto+return_route,contexto+return_route,{},102,0.6493,0.0188,NaN
30,21:49,contexto+coord_salida+return_route,contexto+coord_salida+return_route,{},119,0.6556,0.0221,NaN
31,21:53,base,-,{},88,0.6513,0.0204,NaN
32,21:56,base+coord_salida,coord_salida,{},105,0.6528,0.0203,NaN
33,21:58,base+return_route,return_route,{},96,0.6483,0.0253,NaN
34,21:59,contexto+coord_salida,contexto+coord_salida,{},111,0.6522,0.0201,NaN
35,22:01,contexto+return_route,contexto+return_route,{},102,0.6493,0.0188,NaN
36,22:02,contexto+coord_salida+return_route,contexto+coord_salida+return_route,{},119,0.6556,0.0221,NaN
37,22:30,ref_contexto,contexto,{},94,0.6550,0.0230,NaN


In [22]:

experimento(
    "ctx_param_conservador",
    ["contexto"],
    {
        "num_leaves": 15,
        "min_child_samples": 80,
        "learning_rate": 0.03,
        "n_estimators": 600,
    }
)

experimento(
    "ctx_param_base_leaves31",
    ["contexto"],
    {
        "num_leaves": 31,
        "min_child_samples": 100,
        "learning_rate": 0.03,
        "n_estimators": 600,
    }
)

experimento(
    "ctx_param_mas_capacidad",
    ["contexto"],
    {
        "num_leaves": 63,
        "min_child_samples": 100,
        "learning_rate": 0.03,
        "n_estimators": 600,
    }
)


ctx_param_conservador        CV top-3 = 0.6465  (folds: 0.673 0.665 0.645 0.614 0.634)  94 features  98s
ctx_param_base_leaves31      CV top-3 = 0.6565  (folds: 0.699 0.657 0.652 0.628 0.648)  94 features  110s
ctx_param_mas_capacidad      CV top-3 = 0.6509  (folds: 0.683 0.666 0.634 0.631 0.641)  94 features  145s


(array([-5.02105012, -3.45871919, -2.73926071, ..., -4.50379445,
        -2.55333145, -2.24175536], shape=(474688,)),
 sample_id
 0        1
 1        1
 2        1
 3        1
 4        1
         ..
 15033    1
 15034    1
 15035    1
 15036    1
 15037    1
 Name: target_index, Length: 13175, dtype: int64)

## 6. Dónde falla
Acierto según **el tipo de la tarea correcta**. Útil para decidir qué features atacar (y para Pau).

In [23]:
def por_tipo(h):
    y = lab[lab.target_index == 1].set_index("sample_id").task_index.map(lambda t: TYPES[t])
    return (pd.DataFrame({"tipo": y.reindex(h.index), "hit": h})
              .groupby("tipo").hit.agg(acierto="mean", muestras="size")
              .sort_values("muestras", ascending=False).round(3))

pd.concat({
    "base": por_tipo(h_base),
    "coord_salida": por_tipo(h_coord),
    "return_route": por_tipo(h_route),
    "contexto+coord": por_tipo(h_context_coord),
    "contexto+ruta": por_tipo(h_context_route),
    "contexto+ambos": por_tipo(h_context_both),
}, axis=1)

base          coord_salida           \
                           acierto muestras      acierto muestras   
tipo                                                                
transfer_aircraft            0.855     3527        0.864     3527   
assume_aircraft              0.827     3366        0.819     3366   
zone_conflict                0.713     2121        0.715     2121   
entry_coordination           0.353     1068        0.367     1068   
return_to_route              0.513      937        0.511      937   
exit_coordination            0.097      745        0.091      745   
exit_conditions              0.194      448        0.181      448   
conflict_resolution          0.315      390        0.346      390   
qos_improvement              0.167      257        0.167      257   
exit_conflict_resolution     0.142      197        0.157      197   
non_conformance_resolution   0.640       86        0.558       86   
entry_conflict_resolution    0.065       31        0.065       31   
entry_conditions             0.500        2        0.500        2   

                           return_route          contexto+coord           \
                                acierto muestras        acierto muestras   
tipo                                                                       
transfer_aircraft                 0.851     3527          0.851     3527   
assume_aircraft                   0.827     3366          0.827     3366   
zone_conflict                     0.709     2121          0.709     2121   
entry_coordination                0.364     1068          0.363     1068   
return_to_route                   0.479      937          0.511      937   
exit_coordination                 0.093      745          0.111      745   
exit_conditions                   0.185      448          0.214      448   
conflict_resolution               0.333      390          0.333      390   
qos_improvement                   0.179      257          0.148      257   
exit_conflict_resolution          0.142      197          0.183      197   
non_conformance_resolution        0.651       86          0.558       86   
entry_conflict_resolution         0.065       31          0.129       31   
entry_conditions                  0.000        2          0.000        2   

                           contexto+ruta          contexto+ambos           
                                 acierto muestras        acierto muestras  
tipo                                                                       
transfer_aircraft                  0.862     3527          0.861     3527  
assume_aircraft                    0.825     3366          0.828     3366  
zone_conflict                      0.696     2121          0.716     2121  
entry_coordination                 0.377     1068          0.378     1068  
return_to_route                    0.477      937          0.494      937  
exit_coordination                  0.101      745          0.099      745  
exit_conditions                    0.196      448          0.212      448  
conflict_resolution                0.313      390          0.328      390  
qos_improvement                    0.175      257          0.167      257  
exit_conflict_resolution           0.132      197          0.162      197  
non_conformance_resolution         0.593       86          0.651       86  
entry_conflict_resolution          0.097       31          0.065       31  
entry_conditions                   0.000        2          0.000        2

## 7. Modelo final: guardar OOF, modelo y predicciones
Cuando tengas la mejor combinación, pon aquí `BEST_GROUPS` y `BEST_PARAMS` y ejecuta. Genera:
- `oof/lgb_adri.csv` — predicciones OOF (para el stacking de Juanma y el análisis de Pau)
- `models/lgb_adri.txt` — **los pesos**: el modelo entrenado con todo train+validation
- `preds/lgb_adri.csv` y `submissions/sub_lgb_adri.csv` — predicciones de test

In [24]:
# BEST_GROUPS = list(FEATURE_GROUPS)   # <- cambiar por la mejor combinación
# BEST_PARAMS = {}                     # <- parámetros cambiados respecto a PARAMS

# params = {**PARAMS, **BEST_PARAMS}
# d, F = build(lab, BEST_GROUPS)
# oof, h, _ = cv(d, F, params)
# print(f"CV top-3 final: {h.mean():.4f}")
# d[["sample_id", "row_index"]].assign(score=oof).to_csv(ROOT / "oof" / f"{NAME}.csv", index=False)

# m = fit(d, F, params)
# m.booster_.save_model(str(ROOT / "models" / f"{NAME}.txt"))

# dt, _ = build(test, BEST_GROUPS)
# pred = dt[["sample_id", "row_index"]].assign(score=m.predict(dt[F]))
# pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)
# sub = (pred.sort_values(["sample_id", "score"], ascending=[True, False]).groupby("sample_id").row_index
#            .apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction").reset_index())
# sub.to_csv(ROOT / "submissions" / f"sub_{NAME}.csv", index=False)
# print("Guardado:", NAME, "| test:", len(sub), "muestras")

El chat me ha dicho que eso es un poco mierda porque los mete todos y eso baja el promedio. Me recomienda que haga esto:

In [25]:

# Seleccionar únicamente grupos cuya mejora se haya comprobado.
# De momento, contexto es la mejor referencia documentada (CV 0.6551).
BEST_GROUPS = ["contexto"]

# Mantener los parámetros base hasta comprobar si otros mejoran la CV.
BEST_PARAMS = {}

params = {**PARAMS, **BEST_PARAMS}

# 1. Preparar los datos con las características elegidas.
d, F = build(lab, BEST_GROUPS)

# 2. Evaluar de nuevo con los cinco folds.
oof, h, por_fold = cv(d, F, params)
print(f"CV top-3 final: {h.mean():.4f}")
print("CV por fold:", por_fold.round(4).to_dict())

# 3. Guardar las predicciones OOF para el análisis y stacking.
d[["sample_id", "row_index"]].assign(score=oof).to_csv(
    ROOT / "oof" / f"{NAME}.csv", index=False
)

# 4. Entrenar el modelo final con todos los datos etiquetados.
m = fit(d, F, params)
m.booster_.save_model(str(ROOT / "models" / f"{NAME}.txt"))

# 5. Aplicar exactamente las mismas características al test.
dt, _ = build(test, BEST_GROUPS)
pred = dt[["sample_id", "row_index"]].assign(
    score=m.predict(dt[F])
)
pred.to_csv(ROOT / "preds" / f"{NAME}.csv", index=False)

# 6. Seleccionar los tres row_index con mayor puntuación por muestra.
sub = (
    pred.sort_values(
        ["sample_id", "score"], ascending=[True, False]
    )
    .groupby("sample_id")
    .row_index
    .apply(lambda r: " ".join(map(str, r.iloc[:3])))
    .rename("prediction")
    .reset_index()
)

sub.to_csv(
    ROOT / "submissions" / f"sub_{NAME}.csv", index=False
)
print("Guardado:", NAME, "| test:", len(sub), "muestras")


CV top-3 final: 0.6550
CV por fold: {0: 0.6891, 1: 0.6565, 2: 0.6593, 3: 0.6266, 4: 0.644}
Guardado: lgb_coord | test: 1863 muestras


### Comprobación con el scorer oficial
Con el OOF construimos un envío solo para las muestras de `validation.csv` y lo puntúa `Data/scorer.py`. Debe dar lo mismo que nuestro `hit3` en esas muestras.

In [26]:
val_ids = set(pd.read_csv(DATA / "validation.csv", usecols=["sample_id"]).sample_id)
o = d[["sample_id", "row_index"]].assign(score=oof)
o = o[o.sample_id.isin(val_ids)].sort_values(["sample_id", "score"], ascending=[True, False])
o.groupby("sample_id").row_index.apply(lambda r: " ".join(map(str, r.iloc[:3]))).rename("prediction") \
 .reset_index().to_csv(ROOT / "submissions" / f"val_{NAME}.csv", index=False)
print(f"hit3 propio en validation: {h[h.index.isin(val_ids)].mean():.4f}")
r = subprocess.run([sys.executable, DATA / "scorer.py", "--labelled", DATA / "validation.csv",
                    ROOT / "submissions" / f"val_{NAME}.csv"], capture_output=True, text=True)
print("scorer oficial:", r.stdout or r.stderr)

hit3 propio en validation: 0.6709
scorer oficial: top-3 hit rate: 0.6709  (1252 samples)



In [28]:

# Generar y guardar las predicciones OOF de LightGBM
# con la configuración que queremos evaluar en el ensemble.

BEST_GROUPS = ["contexto"]
BEST_PARAMS = {}

params = {**PARAMS, **BEST_PARAMS}

d_lgb, F_lgb = build(lab, BEST_GROUPS)
oof_lgb, h_lgb, folds_lgb = cv(d_lgb, F_lgb, params)

print(f"CV top-3 LightGBM: {h_lgb.mean():.4f}")
print("CV por fold:", folds_lgb.round(4).to_dict())

lgb_oof = d_lgb[["sample_id", "row_index"]].copy()
lgb_oof["score"] = oof_lgb

out_path = ROOT / "oof" / "lgb_adri_ctx.csv"
lgb_oof.to_csv(out_path, index=False)

print("OOF guardado en:", out_path)
print("Número de filas:", len(lgb_oof))


# Unir las predicciones OOF de ambos modelos.
keys = ["sample_id", "row_index"]

truth = lab[keys + ["target_index", "fold"]].copy()

lgb_oof = pd.read_csv(
    ROOT / "oof" / "lgb_adri_ctx.csv"
).rename(columns={"score": "score_lgb"})

nn_oof = pd.read_csv(
    ROOT / "oof" / "nn_tf_v2.csv"
).rename(columns={"score": "score_nn"})

z = (
    truth
    .merge(lgb_oof, on=keys, validate="one_to_one")
    .merge(nn_oof, on=keys, validate="one_to_one")
)

assert len(z) == len(truth), "Faltan o sobran predicciones OOF"

# Convertir las puntuaciones a rangos relativos dentro de cada snapshot.
z["rank_lgb"] = z.groupby("sample_id")["score_lgb"].rank(pct=True)
z["rank_nn"] = z.groupby("sample_id")["score_nn"].rank(pct=True)

folds = (
    z.drop_duplicates("sample_id")
     .set_index("sample_id")["fold"]
)

# Probar cuánto peso dar a la red neuronal.
for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]:
    z["score_blend"] = (
        (1 - w) * z["rank_lgb"] +
        w * z["rank_nn"]
    )

    top3 = (
        z.sort_values(
            ["sample_id", "score_blend"],
            ascending=[True, False]
        )
        .groupby("sample_id")
        .head(3)
    )

    hits = top3.groupby("sample_id")["target_index"].max()
    per_fold = hits.groupby(folds.reindex(hits.index)).mean()

    print(
        f"Peso NN={w:.1f} | CV={hits.mean():.4f} | "
        f"folds={per_fold.round(3).to_dict()}"
    )


CV top-3 LightGBM: 0.6550
CV por fold: {0: 0.6891, 1: 0.6565, 2: 0.6593, 3: 0.6266, 4: 0.644}
OOF guardado en: c:\Users\Pauet\Paueti\Hackathones\Engage2 2026\FlowDataLab\oof\lgb_adri_ctx.csv
Número de filas: 474688
Peso NN=0.0 | CV=0.6550 | folds={0: 0.689, 1: 0.657, 2: 0.659, 3: 0.627, 4: 0.644}
Peso NN=0.1 | CV=0.6559 | folds={0: 0.692, 1: 0.658, 2: 0.659, 3: 0.628, 4: 0.644}
Peso NN=0.2 | CV=0.6567 | folds={0: 0.699, 1: 0.657, 2: 0.658, 3: 0.627, 4: 0.644}
Peso NN=0.3 | CV=0.6600 | folds={0: 0.705, 1: 0.66, 2: 0.659, 3: 0.631, 4: 0.647}
Peso NN=0.4 | CV=0.6575 | folds={0: 0.704, 1: 0.654, 2: 0.66, 3: 0.625, 4: 0.645}
Peso NN=0.5 | CV=0.6559 | folds={0: 0.703, 1: 0.653, 2: 0.66, 3: 0.623, 4: 0.643}


### Cómo usan los demás el modelo guardado (sin entrenar)
```python
m = lgb.Booster(model_file="models/lgb_adri.txt")
dt, F = build(test, BEST_GROUPS)        # mismas features que en el entrenamiento
score = m.predict(dt[F])
```